# Labeling
We label the data collected in [01_data_collection.ipynb](01_data_collection.ipynb) using zero-shot classification via `facebook/bart-large-mnli`.

In [1]:
from transformers import pipeline
import torch

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0,
    dtype=torch.float16,
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

We define some `relevant` categories and some `irrelevant` categories.

In [2]:
relevant = {
    "Monetary and fiscal policy",
    "Macroeconomic data and indicators",
    "Geopolitics",
    "International trade",
    "Corporate and sector dynamics",
    "Commodities and supply Chain",
    "Systemic shocks",
    "Natural environment shocks",
    "Public health",
    "Technology advancements",
}

irrelevant = {
    "Entertainment",
    "Culture and art",
    "Sports",
    "Lifestyle, travel, wellness",
    # "Local news",
    "Science and pure research",
    "Long-form Features & Investigative Essays",
    "Personal & Human Interest",
}

all_labels = list(relevant | irrelevant)

We label sequences according to the given relevant/irrelevant classes.

In [3]:
import numpy as np


def relevant_labeler(batch):
    results = classifier(batch["concat"], candidate_labels=all_labels)

    labels = []
    for res in results:
        scores = dict(zip(res["labels"], res["scores"]))

        top_label = res["labels"][0]
        top_score = res["scores"][0]
        rel_sum = sum(scores[cat] for cat in relevant)

        if top_score > 0.5:
            label = 1 if top_label in relevant else 0
        elif rel_sum > 0.65:
            label = 1
        elif rel_sum < 0.35:
            label = 0
        else:
            label = -1

        labels.append(label)

    return {"relevant": labels}

We now load, clean, and label the `raw_headlines.csv` dataset.

We first merge title and description into a HuggingFace `Dataset` for faster processing.

In [4]:
import pandas as pd
from datasets import Dataset

df_rel = pd.read_csv("data/raw_headlines.csv")
df_irrel = pd.read_csv("data/irrelevant_headlines.csv")
df = pd.concat([df_rel, df_irrel], ignore_index=True)

df["concat"] = df["title"] + " | " + df["description"]
df.dropna(inplace=True)
ds = Dataset.from_pandas(df[["concat"]])

We apply the labeling function: relevant (`1`), irrelevant (`0`), or unknown (`-1`).

In [5]:
ds = ds.map(relevant_labeler, batched=True, batch_size=128)

Map:   0%|          | 0/3605 [00:00<?, ? examples/s]

and copy the new columns in the original `pandas` dataframe:

In [6]:
df.dropna(inplace=True)
df["relevant"] = ds["relevant"]

In [7]:
df.to_csv("data/labeled_headlines.csv", index=False)